In [1]:
!pip -q install "transformers>=4.43.0" "peft==0.13.2" sentencepiece

from google.colab import drive
drive.mount('/content/drive')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 10.7 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
import os, re, json, urllib.parse, requests
from typing import List, Tuple, Dict, Any, Optional

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ======= 경로/모델 이름 =======
# 학습 때 save_model(OUTPUT_DIR)로 저장했던 LoRA 디렉터리 지정
LORA_DIR   = "/content/drive/MyDrive/eng_exaone_rewrite_inaproppriate_lora_eos/checkpoint-212/"  # ← 필요시 수정
BASE_MODEL = "LGAI-EXAONE/EXAONE-4.0-1.2B"

# 출력 파일 경로
OUT_LOCAL  = "/content/rewritten_dev.json"
OUT_DRIVE  = "/content/drive/MyDrive/eng_exaone_rewrite_inaproppriate_lora_eos/rewritten_inaproppriate_dev.json"

# 디바이스/DTYPE
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    DTYPE = torch.bfloat16
else:
    DTYPE = torch.float16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE, device


(torch.bfloat16, device(type='cuda'))

In [3]:
# --- 최신 버전으로 업데이트 (중요) ---
!pip -q install -U "transformers>=4.45.0" "accelerate>=0.34.0" "tokenizers>=0.19.1" "safetensors" "sentencepiece"

import torch, json, os
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# dtype 선택
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    DTYPE = torch.bfloat16
else:
    DTYPE = torch.float16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1) Config 를 먼저 remote code로 로드
config = AutoConfig.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,     # ★ 커스텀 아키텍처 exaone4 지원
)

# 2) Tokenizer: fast 실패 가능 → slow로 폴백
def load_tokenizer_with_fallback(model_name: str):
    try:
        tok = AutoTokenizer.from_pretrained(
            model_name, use_fast=True, trust_remote_code=True
        )
        print("[OK] fast tokenizer loaded")
        return tok
    except Exception as e:
        print("[WARN] fast tokenizer failed → slow fallback:", repr(e))
        tok = AutoTokenizer.from_pretrained(
            model_name, use_fast=False, trust_remote_code=True
        )
        print("[OK] slow tokenizer loaded")
        return tok

tokenizer = load_tokenizer_with_fallback(BASE_MODEL)

# pad/eos 안전장치
changed = False
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else "[PAD]"
    changed = True
if tokenizer.eos_token_id is None:
    tokenizer.add_special_tokens({"eos_token": "</s>"})
    changed = True

# 3) Base model 로드 (remote code 신뢰)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    config=config,
    torch_dtype=DTYPE,
    trust_remote_code=True,     # ★ 필수
)

# 토큰 추가 시 임베딩 리사이즈
if changed:
    base_model.resize_token_embeddings(len(tokenizer))

# 4) LoRA 어댑터 적용
model = PeftModel.from_pretrained(base_model, LORA_DIR)
model.config.use_cache = False
model.to(device).eval()

print("✅ Loaded:", getattr(model.config, "model_type", None))
print("pad_token_id:", tokenizer.pad_token_id, "| eos_token_id:", tokenizer.eos_token_id)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


[OK] fast tokenizer loaded


model.safetensors:   0%|          | 0.00/2.56G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

✅ Loaded: exaone4
pad_token_id: 0 | eos_token_id: 361


In [4]:
def to_raw(url: str) -> str:
    parsed = urllib.parse.urlparse(url)
    parts = parsed.path.strip('/').split('/')
    if len(parts) >= 5 and parts[2] == 'blob':
        owner, repo, _, branch = parts[:4]
        path_rest = '/'.join(parts[4:])
        return f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{path_rest}"
    return url

SRC_URL = "https://github.com/beefed-up-geek/HCLT-KACL-2025/blob/main/Korean_Inappropriate_Detection/dataset/original_english_formatted/dev.json"
RAW_URL = to_raw(SRC_URL)

resp = requests.get(RAW_URL, timeout=30)
resp.raise_for_status()
items = resp.json()
print("Loaded items:", len(items))
print(items[0].keys(), "\n")
print(items[0]["dialogue"][:200])


Loaded items: 101
dict_keys(['id', 'dialogue', 'output']) 

Speaker2: But from a national perspective, it's something that really weakens our power. Not having kids just eats away at our productivity.
Speaker1: Why do those idiots keep saying there's no future


In [ ]:
def to_pairs(dialogue: str) -> List[Tuple[str, str]]:
    pairs = []
    for ln in [ln.strip() for ln in dialogue.split("\n") if ln.strip()]:
        m = re.match(r"^(speaker\d+)\s*:\s*(.*)$", ln)
        if m: pairs.append((m.group(1), m.group(2)))
        else: pairs.append(("speaker?", ln))
    return pairs

def build_user_block(context_pairs: List[Tuple[str, str]], target_text: str) -> str:
    # 학습 포맷과 동일한 [dialogue]/[rewrite]
    dialogue_block = "\n".join(f"{spk}: {txt}" for spk, txt in context_pairs)
    return f"""[dialogue]
{dialogue_block}
[rewrite]
{target_text}
"""

# 시스템 프롬프트(힌트는 생략 가능—카테고리별 힌트가 있다면 여기에 추가할 수도 있음)
def build_system_prompt() -> str:
    header = (
        "You are an expert at rewriting dialogue.\n"
        "Rewrite the final utterance in the given conversation.\n"
        "[Rules]\n"
    )
    rules = [
        "- First, understand the given context.\n",
        "- Keep the meaning similar but improve readability by fixing spelling/grammar, using standard expressions, and refining wording.\n",
        "- If the sentence is incomplete, complete it into a full sentence.\n",
        "- Output only the single rewritten line with no extra commentary.\n",
    ]
    return header + "".join(rules)

# 학습 당시 assistant 시작 접두사(라벨 시작점)
RESPONSE_TMPL = '{"role":"assistant","content":"'


In [ ]:
import json
import torch

@torch.no_grad()
def rewrite_one_turn(
    all_pairs: list[tuple[str, str]],
    turn_idx: int,
    max_new_tokens: int = 64
) -> str:
    # 컨텍스트는 해당 턴까지 포함
    ctx_pairs = all_pairs[:turn_idx + 1]
    target = all_pairs[turn_idx][1]  # 해당 턴의 원문 텍스트

    # ✅ 카테고리 힌트가 포함된 시스템 프롬프트
    sys_prompt = build_system_prompt()
    user_prompt = build_user_block(ctx_pairs, target)

    system_line = json.dumps(
        {"role": "system", "content": sys_prompt},
        ensure_ascii=False, separators=(',', ':')
    )
    user_line = json.dumps(
        {"role": "user", "content": user_prompt},
        ensure_ascii=False, separators=(',', ':')
    )

    # 학습 포맷과 동일하게 assistant 시작 접두사까지만 넣고 생성
    prompt = f"{system_line}\n{user_line}\n{RESPONSE_TMPL}"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    gen = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,  # EOS에서 정지
        no_repeat_ngram_size=3,
        repetition_penalty=1.05,
    )

    out = tokenizer.decode(
        gen[0][inputs["input_ids"].size(1):],
        skip_special_tokens=True
    )

    # EOS 앞까지만 안전하게 슬라이스
    if tokenizer.eos_token:
        out = out.split(tokenizer.eos_token)[0]

    # 끝에 남을 수 있는 따옴표/괄호 잔재 간단 정리
    out = out.strip().rstrip('"').rstrip("}").strip()
    return out


In [ ]:
def count_words_korean(s: str) -> int:
    # 간단 공백 기준 단어 수
    return len([w for w in s.strip().split() if w])

def rewrite_dialogue(dialogue: str) -> str:
    pairs = to_pairs(dialogue)
    #if len(pairs) <= 2:
    #    # 3턴 미만이면 변경 없이 반환
    #    return dialogue

    new_pairs = pairs[:]  # 얕은 복사
    for i in range(2, len(pairs)):  # ✅ 3번째 턴부터 재작성
        spk, orig = pairs[i]
        try:
            rew = rewrite_one_turn(pairs, i, max_new_tokens=1024).strip()
        except Exception:
            # 추론 실패 시 원본 유지
            rew = orig

        # 길이 규칙 적용: 원본≥5 단어 & 재작성 > 1.5배면 [err] + 원본으로 대체
        orig_words = count_words_korean(orig)
        rew_words  = count_words_korean(rew)
        if orig_words >= 5 and rew_words > int(1.5 * orig_words):
            new_pairs[i] = (spk, f"[err] {orig}")
        else:
            new_pairs[i] = (spk, rew)

    return "\n".join(f"{spk}: {txt}" for spk, txt in new_pairs)


In [ ]:
from tqdm.auto import tqdm
import os, json

os.makedirs(os.path.dirname(OUT_DRIVE), exist_ok=True)

rewritten = []
for ex in tqdm(items, total=len(items), desc="Rewriting dialogues", dynamic_ncols=True):
    d0 = ex.get("dialogue", "")

    d1 = rewrite_dialogue(d0)
    new_ex = dict(ex)
    new_ex["dialogue_orig"] = d0
    new_ex["dialogue"] = d1
    rewritten.append(new_ex)

with open(OUT_LOCAL, "w", encoding="utf-8") as f:
    json.dump(rewritten, f, ensure_ascii=False, indent=2)

with open(OUT_DRIVE, "w", encoding="utf-8") as f:
    json.dump(rewritten, f, ensure_ascii=False, indent=2)

print(f"Saved: {OUT_LOCAL}")
print(f"Saved: {OUT_DRIVE}")
print("Preview:\n", rewritten[0]["dialogue"][:300])


Rewriting dialogues:   0%|          | 0/101 [00:00<?, ?it/s]

Saved: /content/rewritten_dev.json
Saved: /content/drive/MyDrive/eng_exaone_rewrite_inaproppriate_lora_eos/rewritten_inaproppriate_dev.json
Preview:
 speaker?: Speaker2: But from a national perspective, it's something that really weakens our power. Not having kids just eats away at our productivity.
speaker?: Speaker1: Why do those idiots keep saying there's no future for gay people when they aren't having kids themselves?
speaker?: I think femin


In [5]:
'''
messages = [
    {"role": "system", "content": (
        "You are an expert at rewriting dialogue.\n"
        "Rewrite the final utterance in the given conversation.\n"
        "[Rules]\n"
        "- First, understand the given context.\n"
        "- Keep the meaning similar but improve readability by fixing spelling/grammar, using standard expressions, and refining wording.\n"
        "- If the sentence is incomplete, complete it into a full sentence.\n"
        "- Output only the single rewritten line with no extra commentary.\n"
    )},
    {"role": "user", "content": (
        "[dialogue]"
        "Speaker2: But from a national perspective, it's something that really weakens our power. Not having kids just eats away at our productivity.\nSpeaker1: Why do those idiots keep saying there's no future for gay people when they aren't having kids themselves?\nSpeaker2: It's the feminists gaslighting guys that's making society fall apart."
        "[rewrite]"
        "It's the feminists gaslighting guys that's making society fall apart."
    )}
]
input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

output = model.generate(
    input_ids.to(model.device),
    max_new_tokens=128,
    do_sample=False,
)
print(tokenizer.decode(output[0]))
'''

[|system|]
You are an expert at rewriting dialogue.
Rewrite the final utterance in the given conversation.
[Rules]
- First, understand the given context.
- Keep the meaning similar but improve readability by fixing spelling/grammar, using standard expressions, and refining wording.
- If the sentence is incomplete, complete it into a full sentence.
- Output only the single rewritten line with no extra commentary.
[|endofturn|]
[|user|]
[dialogue]Speaker2: But from a national perspective, it's something that really weakens our power. Not having kids just eats away at our productivity.
Speaker1: Why do those idiots keep saying there's no future for gay people when they aren't having kids themselves?
Speaker2: It's the feminists gaslighting guys that's making society fall apart.[rewrite]It's the feminists gaslighting guys that's making society fall apart.[|endofturn|]
[|assistant|]
<think>

</think>

Feminists are gaslighting men, and that's why society's falling apart.[|endofturn|]
